2.1 Separación de datos:
    80:20 split-> low amount of data, recommended split
    randomly selected

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load preprocessed dataset
df = pd.read_csv('insurance_cleaned.csv')

# Separate features (X) and target variable (y)
X = df.drop(columns=['charges'])
y = df['charges']

# Perform 80:20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=69
)

2.2 Validación cruzada
    5-fold cross validation, only with training set: we will use z-score scaling(on train)

In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline(
    [('scaler', StandardScaler()), ('model', LinearRegression())]
)

kf = KFold(n_splits=5, shuffle=True, random_state=69)

# Perform 5-fold cross-validation
cv_results = cross_validate(
    pipeline,
    X_train,
    y_train,
    cv=kf,
    scoring=['neg_root_mean_squared_error', 'r2'],
    return_train_score=True,
)

print(
    f"CV Validation RMSE: {-cv_results['test_neg_root_mean_squared_error'].mean():.4f}"
    f" (± {cv_results['test_neg_root_mean_squared_error'].std():.4f})"
)
print(
    f"CV Validation R²:   {cv_results['test_r2'].mean():.4f} (±"
    f" {cv_results['test_r2'].std():.4f})"
)

CV Validation RMSE: 5923.0506 (± 207.0523)
CV Validation R²:   0.7553 (± 0.0208)


2.3 Entrenamiento del modelo
    regresión lineal+ print out error de train y de validación utilizando el RMS

In [14]:
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

# Fit the pipeline on the full training set
pipeline.fit(X_train, y_train)

# Predictions
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)

# Compute RMSE and R²
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_val = np.sqrt(mean_squared_error(y_test, y_test_pred))

r2_train = r2_score(y_train, y_train_pred)
r2_val = r2_score(y_test, y_test_pred)

print(f"Train RMSE:      {rmse_train:.4f} | Train R²:      {r2_train:.4f}")
print(f"Validation RMSE: {rmse_val:.4f} | Validation R²: {r2_val:.4f}")

Train RMSE:      5895.5082 | Train R²:      0.7585
Validation RMSE: 6676.1316 | Validation R²: 0.7160
